# Text Re-identification Evaluation with Retrieval-Augmented Generation

# Initialization

## Imports

In [20]:
import os, csv, json
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
from datetime import datetime
from typing import Optional, List
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import random
from collections import OrderedDict

import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig

from langchain_core.documents import Document as LangchainDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy

from cappr.huggingface.classify import cache_model, predict_proba_examples
from cappr import Example

## Settings

In [21]:
#region Input

ID_KEY = "doc_id"
TEXT_KEY = "text"
BK_KEY = "background_knowledge"

# Dataset
DATASET_NAME = "wiki553"
DATASET_FOLDER_PATH = f"data/{DATASET_NAME}"
CORPUS_FILE_PATH = f"{DATASET_FOLDER_PATH}/corpora/Wiki553_Corpus.json"
ANONYMIZATION_PREFIX = f"{DATASET_FOLDER_PATH}/anonymizations/Wiki553_"
ANONYMIZATIONS_FILE_PATHS = {
    "St.NER3":f"{ANONYMIZATION_PREFIX}St.NER3.json",
    "St.NER4":f"{ANONYMIZATION_PREFIX}St.NER4.json",
    "St.NER7":f"{ANONYMIZATION_PREFIX}St.NER7.json",
    "spaCy":f"{ANONYMIZATION_PREFIX}spaCy.json",
    "Presidio":f"{ANONYMIZATION_PREFIX}Presidio.json",        
    "Word2Vec_t=0.5":f"{ANONYMIZATION_PREFIX}Word2Vec_t=0.5.json",
    "Word2Vec_t=0.25":f"{ANONYMIZATION_PREFIX}Word2Vec_t=0.25.json",
    "k-anonymity_Random":f"{ANONYMIZATION_PREFIX}k-anonymity_Random.json",
    "k-anonymity_Greedy":f"{ANONYMIZATION_PREFIX}k-anonymity_Greedy.json",
    "Manual":f"{ANONYMIZATION_PREFIX}Manual.json",
    "Student-LLM":f"{ANONYMIZATION_PREFIX}gpt-4o-2024-09-03_Student.json",
    "Sparks-LLM":f"{ANONYMIZATION_PREFIX}gpt-4o-2024-09-03_Sparks.json",        
    "MvM-LLM":f"{ANONYMIZATION_PREFIX}gpt-4o-2024-09-03_MvM.json",     
}
BK_FILE_PATH = "data/wiki553/bks/Wiki553_BK=Public.json"

# Background knowledge
USE_NO_BK = False
bk_file_name = os.path.basename(BK_FILE_PATH).replace(".json", "")
BK_NAME = "NoBK" if USE_NO_BK else bk_file_name

#endregion


#region Retriever

RETRIEVER_K = 10
RETRIEVER_CHUNK_SIZE = 128
RETRIEVER_NAME = f"k={RETRIEVER_K}_chunk={RETRIEVER_CHUNK_SIZE}"
RETRIEVER_USE_DENSE = True # Dense=FAISS | Sparse=BM25
RETRIEVER_NAME += "_FAISS" if RETRIEVER_USE_DENSE else "_BM25"
RETRIEVER_MARKDOWN_SEPARATORS = [
    "\n#{1,6} ",
    "```\n",
    "\n\\*\\*\\*+\n",
    "\n---+\n",
    "\n___+\n",
    "\n\n",
    "\n",
    " ",
    "",
]
RETRIEVER_SPLIT_CHUNK_OVERLAP_DIVISOR = 10
RETRIEVER_REMOVE_MASKING_MARKS = True
if RETRIEVER_REMOVE_MASKING_MARKS:
    RETRIEVER_NAME += "_NoMasks"
RETRIEVER_MASKING_MARKS = ["SENSITIVE", "PERSON", "DEM", "LOC",
                 "ORG", "DATETIME", "QUANTITY", "MISC",
                 "NORP", "FAC", "GPE", "PRODUCT", "EVENT",
                 "WORK_OF_ART", "LAW", "LANGUAGE", "DATE",
                 "TIME", "ORDINAL", "CARDINAL", "DATE_TIME", "DATETIME",
                 "NRP", "LOCATION", "ORGANIZATION", "\*\*\*"]
RETRIEVER_EMBEDDING_MODEL_NAME = "thenlper/gte-small" # For dense retriever
RETRIEVER_EMBEDDING_BATCH_SIZE = 32
RETRIEVER_NAME += f"_{RETRIEVER_EMBEDDING_MODEL_NAME}"

#endregion


#region Reader

READER_MODEL_NAME = "unsloth/Qwen3-4B-bnb-4bit"
READER_MODEL_CONFIG = AutoConfig.from_pretrained(READER_MODEL_NAME)
READER_QUANTIZATION_BITS = None
if READER_QUANTIZATION_BITS == 4:
    READER_QUANTIZATION_CONFIG = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
elif READER_QUANTIZATION_BITS == 8:
    READER_QUANTIZATION_CONFIG = BitsAndBytesConfig(load_in_8bit=True)
else:
    READER_QUANTIZATION_CONFIG = None

READER_BATCH_SIZE = 1
READER_PROMPT_0_USER = [ # Only User role
    {
        "role": "user",
        "content": """Task: Predict the ID of the person corresponding to a document.

Input:
BACKGROUND: A list of people and their details, indexed by ID.
DOCUMENT: A text related to one person from the BACKGROUND.
NAMES: A summary list of the IDs from people in BACKGROUND.

Output:
The ID of the person that best fits the DOCUMENT based on the BACKGROUND. Important: No reasoning, directly return the ID.

----------------

BACKGROUND:{background}


DOCUMENT: {document}


NAMES: {names}

----------------

Based on the input provided, the ID corresponding to the document is:"""
    },
]
READER_PROMPT_1_USER_ASSISTANT = [ # User, Assistant roles
    {
        "role": "user",
        "content": """Task: Predict the ID of the person corresponding to a document.

Input:
BACKGROUND: A list of people and their details, indexed by ID.
DOCUMENT: A text related to one person from the BACKGROUND.
NAMES: A summary list of the IDs from people in BACKGROUND.

Output:
The ID of the person that best fits the DOCUMENT based on the BACKGROUND. Important: No reasoning, directly return the ID.

----------------

BACKGROUND:{background}


DOCUMENT: {document}


NAMES: {names}
"""
    },
    {
        "role": "assistant",
        "content": """Based on the input provided, the ID corresponding to the document is:"""
    },
]
READER_PROMPT_2_SYSTEM_USER_ASSISTANT = [ # System, User, Assistant roles
    {
        "role": "system",
        "content": """Task: Predict the ID of the person corresponding to a document.

Input:
BACKGROUND: A list of people and their details, indexed by ID.
DOCUMENT: A text related to one person from the BACKGROUND.
NAMES: A summary list of the IDs from people in BACKGROUND.

Output:
The ID of the person that best fits the DOCUMENT based on the BACKGROUND. Important: No reasoning, directly return the ID.""",
    },
    {
        "role": "user",
        "content": """BACKGROUND:{background}

----------------

DOCUMENT: {document}

----------------

NAMES: {names}

"""
    },
    {
        "role": "assistant",
        "content": """Based on the input provided, the ID corresponding to the document is:"""
    },
]
READER_PROMPT_3_SYSTEM_USER = [ # System, User roles
    {
        "role": "system",
        "content": """Task: Predict the ID of the person corresponding to a document.

Input:
BACKGROUND: A list of people and their details, indexed by ID.
DOCUMENT: A text related to one person from the BACKGROUND.
NAMES: A summary list of the IDs from people in BACKGROUND.

Output:
The ID of the person that best fits the DOCUMENT based on the BACKGROUND. Important: No reasoning, directly return the ID.""",
    },
    {
        "role": "user",
        "content": """BACKGROUND:{background}

----------------

DOCUMENT: {document}

----------------

NAMES: {names}

----------------

Based on the input provided, the ID corresponding to the document is:"""
    },
]
READER_PROMPT_4_SYSTEM_USER = [
    {
        "role": "system",
        "content": """You are an assistant that identifies which person from a given BACKGROUND corresponds to a DOCUMENT.

Your task:
- Read the BACKGROUND, which lists people and their details by ID.
- Read the DOCUMENT, which relates to exactly one person.
- Read NAMES, which lists all available IDs.
- Decide which ID best matches the DOCUMENT.

Output format:
- Respond with only the matching ID, nothing else (no reasoning, no explanation)."""
    },
    {
        "role": "user",
        "content": """BACKGROUND:
{background}

---

DOCUMENT:
{document}

---

NAMES:
{names}

---

Return only the ID that best matches the DOCUMENT:"""
    },
]
READER_PROMPT_5_USER = [
    {
        "role": "user",
        "content": """Your task is to select the ID that best matches the DOCUMENT based on the BACKGROUND.

Inputs:
- BACKGROUND lists people and their details by ID.
- DOCUMENT is related to exactly one person. It might be pseudo-anonymized.
- IDs lists all available identifiers.

Output:
- Respond with only the matching ID. Do not include explanations or reasoning.

---

BACKGROUND:
{background}

---

DOCUMENT:
{document}

---

IDs:
{names}

---

The ID that best matches the DOCUMENT is:"""
    },
]
READER_PROMPT_6_NOBK = [
    {
        "role": "user",
        "content": """Your task is to select the ID that best matches the DOCUMENT.

Inputs:
- DOCUMENT is related to exactly one person. It might be pseudo-anonymized.
- IDs lists all available identifiers.

Output:
- Respond with only the matching ID. Do not include explanations or reasoning.

---

DOCUMENT:
{document}

---

IDs:
{names}

---

The ID that best matches the DOCUMENT is:"""
    },
]
READER_PROMPTS_LIST = [READER_PROMPT_0_USER,
                        READER_PROMPT_1_USER_ASSISTANT,
                        READER_PROMPT_2_SYSTEM_USER_ASSISTANT,
                        READER_PROMPT_3_SYSTEM_USER,
                        READER_PROMPT_4_SYSTEM_USER,
                        READER_PROMPT_5_USER,
                        READER_PROMPT_6_NOBK]
READER_PROMPT_IDX = 6 if USE_NO_BK else 5
READER_SELECTED_PROMPT = READER_PROMPTS_LIST[READER_PROMPT_IDX]
READER_REMOVE_ROLE_END_STR = False
READER_USE_PRIOR_PROBABILITIES = False
READER_USE_CAPPR_CACHE = True
READER_PROMPT_SEPARATOR = "----------------" if READER_PROMPT_IDX < 4 else "---"
READER_USE_SHORT_IDS = False
assert READER_USE_SHORT_IDS == False or USE_NO_BK == False, "Short IDs and No BK cannot be used together."
READER_SHORT_IDS = [chr(65+(i%26)) for i in range(RETRIEVER_K)]
READER_NAME = f"{READER_MODEL_NAME}_Quant={READER_QUANTIZATION_BITS}_Prompt={READER_PROMPT_IDX}_RemoveEnd={READER_REMOVE_ROLE_END_STR}"+ \
    f"_CAPPR_Prior={READER_USE_PRIOR_PROBABILITIES}_Cache={READER_USE_CAPPR_CACHE}_ShortIDs={READER_USE_SHORT_IDS}"

#endregion


#region Output

RESULTS_FILEPATH = f"results_{DATASET_NAME}.csv"
RAG_LINKAGE_NAME = f"RAG-Linkage {BK_NAME} {RETRIEVER_NAME} {READER_NAME}"
print(RAG_LINKAGE_NAME)

#endregion


#region Others

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

#endregion

RAG-Linkage Wiki553_BK=Public k=10_chunk=128_FAISS_NoMasks_thenlper/gte-small unsloth/Qwen3-4B-bnb-4bit_Quant=None_Prompt=5_RemoveEnd=False_CAPPR_Prior=False_Cache=True_ShortIDs=False
cuda


# Dataset loading

In [26]:
def get_masked_text(masked_spans:list, original_text:str) -> str:
    masked_text = ""+original_text
    
    for span in reversed(sorted(masked_spans, key=lambda x:x[0], reverse=False)):
        start_idx = span[0]
        end_idx = span[1]
        if len(span)==3:
            replacement = span[2]
        else: # If there is no replacement, use first masking mark
            replacement = RETRIEVER_MASKING_MARKS[0]
        masked_text = masked_text[:start_idx] + replacement + masked_text[end_idx:]
    
    return masked_text   

In [ ]:
data = {}

# Load corpus (ids and texts)
with open(CORPUS_FILE_PATH, "r") as f:
    corpus = json.load(f)
for doc in corpus:
    id = doc[ID_KEY]
    data[id] = {ID_KEY:id, TEXT_KEY:doc[TEXT_KEY]}

# Load background knowledge
with open(BK_FILE_PATH, "r") as f:
    bk = json.load(f)
for id, bk_text in bk.items():
    data[id][BK_KEY] = bk_text

# Load anonymizations
for anon_name, anon_file_path in ANONYMIZATIONS_FILE_PATHS.items():
    with open(anon_file_path, "r") as f:
        anon = json.load(f)
    for id, masked_spans in anon.items():
        data[id][anon_name] = get_masked_text(masked_spans, data[id][TEXT_KEY])

# Transform into a dataframe
df = pd.DataFrame.from_dict(data.values())

# Remove for memory saving
del corpus
del bk
del anon
del data

# Retriever

## Splitting

In [27]:
# Convert into LangchainDocuments
langchain_docs = []
for idx, row in df.iterrows():
  row_bk = row[BK_KEY]
  if row_bk is not None and row_bk.strip() != "":
    langchain_doc = LangchainDocument(page_content=row_bk, metadata={"source": row[ID_KEY]})
    langchain_docs.append(langchain_doc)

In [28]:
def split_documents(    
    langchain_docs: List[LangchainDocument],
    chunk_size: int,
    tokenizer_name: str = RETRIEVER_EMBEDDING_MODEL_NAME,
) -> List[LangchainDocument]:
    """
    Split documents into chunks of maximum size `chunk_size` tokens and return a list of documents.
    """
    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        AutoTokenizer.from_pretrained(tokenizer_name),
        chunk_size=chunk_size,
        chunk_overlap=int(chunk_size / RETRIEVER_SPLIT_CHUNK_OVERLAP_DIVISOR),
        add_start_index=True,
        strip_whitespace=True,
        separators=RETRIEVER_MARKDOWN_SEPARATORS,
    )

    splitted_docs = []
    for doc in tqdm(langchain_docs):
        splitted_docs += text_splitter.split_documents([doc])

    # Remove duplicates
    unique_docs = {}
    splitted_docs_unique = []
    for doc in splitted_docs:
        if doc.page_content not in unique_docs:
            unique_docs[doc.page_content] = True
            splitted_docs_unique.append(doc)

    return splitted_docs_unique

# Perform splitting
splitted_docs = split_documents(    
    langchain_docs,
    RETRIEVER_CHUNK_SIZE,
    tokenizer_name=RETRIEVER_EMBEDDING_MODEL_NAME,
)

  0%|          | 0/548 [00:00<?, ?it/s]

## Instanciate retriever

In [29]:
if RETRIEVER_USE_DENSE:
    # Embedding model in GPU
    embedding_model = HuggingFaceEmbeddings(
        model_name=RETRIEVER_EMBEDDING_MODEL_NAME,
        model_kwargs={"device": DEVICE},
        encode_kwargs={
            "normalize_embeddings": True, # `True` for cosine similarity
            "batch_size": RETRIEVER_EMBEDDING_BATCH_SIZE,  # Process embeddings in batches
        },
        multi_process=False
    )

    # Create FAISS index
    retriever = FAISS.from_documents(
        splitted_docs,
        embedding_model,
        distance_strategy=DistanceStrategy.COSINE
    )

else:
    # Create BM25 retriever
    retriever = BM25Retriever.from_documents(splitted_docs)

## Functions

In [30]:
def retrieval_df(df, retriever, k)->dict:
  retrievals = {}

  for col_name in df.columns:
      if col_name in [BK_KEY, ID_KEY]: # Exclude unnecesary columns
          continue
      retrievals[col_name] = retrieval_col(df, col_name, retriever, k)

  return retrievals

def retrieval_col(df, col_name, retriever, k)->list:
  retrievals = []
  with tqdm(total=len(df), desc=f"Retrievals for {col_name=}") as pbar:
      for idx, row in df.iterrows():
          text = row[col_name]
          if text is None or text.strip() == "":
            retrievals.append(None)
            continue

          retrieved_docs = retrieval_doc(text, retriever, k)
          retrievals.append(retrieved_docs)
          pbar.update(1)

  return retrievals

def retrieval_doc(text, retriever, k, use_masking_marks_removal:bool=RETRIEVER_REMOVE_MASKING_MARKS):
    if use_masking_marks_removal:
        text = remove_masking_marks(text)
    if type(retriever)==BM25Retriever:
        retriever.k = k # Force the proper k
        result = retriever.invoke(text)
    else: # FAISS
        result = retriever.similarity_search(query=text, k=k)
    return result

def remove_masking_marks(original_text:str, masking_marks:list=RETRIEVER_MASKING_MARKS):
    new_text = original_text
    for mark in masking_marks:
        new_text = new_text.replace(mark, "").strip()
        new_text = ' '.join(new_text.split())
    return new_text

# Reader

In [22]:
# Model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    READER_MODEL_NAME,
    config=READER_MODEL_CONFIG,
    quantization_config=READER_QUANTIZATION_CONFIG,
    dtype="auto",
    device_map="auto"
)
original_forward = model.forward
def cached_forward(*args, **kwargs):
    kwargs["use_cache"] = True
    return original_forward(*args, **kwargs)
model.forward = cached_forward

tokenizer = AutoTokenizer.from_pretrained(READER_MODEL_NAME, trust_remote_code=True)

model_and_tokenizer = (model, tokenizer)

In [23]:
# Prompt
rag_prompt_template = tokenizer.apply_chat_template(
    READER_SELECTED_PROMPT, tokenize=False, add_generation_prompt=False
)
print(rag_prompt_template)

<|im_start|>user
Your task is to select the ID that best matches the DOCUMENT based on the BACKGROUND.

Inputs:
- BACKGROUND lists people and their details by ID.
- DOCUMENT is related to exactly one person. It might be pseudo-anonymized.
- IDs lists all available identifiers.

Output:
- Respond with only the matching ID. Do not include explanations or reasoning.

---

BACKGROUND:
{background}

---

DOCUMENT:
{document}

---

IDs:
{names}

---

The ID that best matches the DOCUMENT is:<|im_end|>



In [24]:
# Testing CAPPR, for the case the model returns NaN (might require a second model loading)
samples = [
    Example(
        prompt="Jodie Foster played",
        completions=["Clarice Starling", "Trinity in The Matrix"],
    ),
    Example(
        prompt="Batman, from Batman: The Animated Series, was played by",
        completions=("Pete Holmes", "Kevin Conroy", "Spongebob!"),
    ),
    Example(
        prompt="Scott Andrew Caan (born August 23, 1976) is an American actor. He currently stars as Detective Danny \"Danno\" Williams in the CBS television series Hawaii Five-0 (2010–present), for which he was nominated for a Golden Globe Award. Caan is also known for his recurring role as manager Scott Lavin in the HBO television series Entourage (2009–2011). He was also a part of 1990s rap group The Whooliganz with The Alchemist. The duo went by the names Mad Skillz and Mudfoot, respectively.",
        completions=['Scott Caan', 'Dwayne Johnson', 'Tom Hanks', 'Patrick Stewart'],
    ),
    Example(
        prompt="Scott Andrew Caan (born August 23, 1976) is an American actor. He currently stars as Detective Danny \"Danno\" Williams in the CBS television series Hawaii Five-0 (2010–present), for which he was nominated for a Golden Globe Award. Caan is also known for his recurring role as manager Scott Lavin in the HBO television series Entourage (2009–2011). He was also a part of 1990s rap group The Whooliganz with The Alchemist. The duo went by the names Mad Skillz and Mudfoot, respectively." \
        f"{tokenizer.eos_token}",
        completions=['Scott Caan', 'Dwayne Johnson', 'Tom Hanks', 'Patrick Stewart'],
    ),    
]
sample_probs = predict_proba_examples(
    samples, model_and_tokenizer=(model, tokenizer), batch_size=READER_BATCH_SIZE
)
for probs in sample_probs:
    probs_str = [f"{p:.3f}" for p in probs]
    print(probs_str)

['0.928', '0.072']
['0.001', '0.999', '0.001']
['0.949', '0.014', '0.031', '0.006']
['0.981', '0.008', '0.011', '0.001']


# Re-identification risk assessment

## Functions

In [16]:
def rag_linkage_df(df:pd.DataFrame, retrievals:dict, id_to_label:dict, prompt_template) -> dict:
  predictions = {}
  model_and_tokenizer = (model, tokenizer)

  # For each column
  for col_name, col_retrievals in retrievals.items():
    col_documents = df[col_name]
    col_probs = rag_linkage_col(col_documents, col_retrievals, model_and_tokenizer, prompt_template, id_to_label)
    predictions[col_name] = col_probs

  return predictions

def rag_linkage_col(col_documents, col_retrievals, model_and_tokenizer, prompt_template, id_to_label) -> np.ndarray:
    col_probs = np.zeros((len(col_documents), len(id_to_label)))
    model_needs_caching = READER_USE_CAPPR_CACHE # Only for first caching
    model, tokenizer = model_and_tokenizer
    
    # Generate all the prompts for pipeline batching
    samples = []
    samples_retrieved_labels = []
    samples_doc_idx = []
    sample_idx = 0    
    for doc_idx, (document, doc_retrievals) in enumerate(zip(col_documents, col_retrievals)):
        if not doc_retrievals is None: # If none, argmax will do the equivalent to random guess
            prompt, ids_in_prompt, retrieved_ids, retrieved_ids_counts = rag_linkage_construct_prompt(doc_retrievals,
                                                                                                       document,
                                                                                                       prompt_template,
                                                                                                       id_to_label,
                                                                                                       tokenizer.eos_token)

            # If only one individual retrieved (and BK is considered), that is the response
            if len(retrieved_ids) == 1 and (not USE_NO_BK):
                label = id_to_label[retrieved_ids[0]]
                col_probs[doc_idx][label] = 1
            # Otherwise, reader prediction required
            else:
                retrieved_labels = [id_to_label[id] for id in retrieved_ids]                
                
                if READER_USE_PRIOR_PROBABILITIES:
                    # Computing prior probabilities based on retrieval counts                    
                    for id, count in retrieved_ids_counts.items():
                        col_probs[doc_idx][id_to_label[id]] += count
                    exps = np.exp(col_probs[doc_idx][retrieved_labels])
                    prior = col_probs[doc_idx][retrieved_labels] = exps / np.sum(exps)
                else:
                    prior = None
                
                if READER_USE_CAPPR_CACHE:
                    # Cache model only for the first time
                    if model_needs_caching:
                        prompt_prefix = prompt.split(READER_PROMPT_SEPARATOR)[0]+READER_PROMPT_SEPARATOR
                        model_and_tokenizer = cache_model(model_and_tokenizer, prompt_prefix)
                        model_needs_caching = False
                    # The rest is the unique part of the prompt
                    prompt = prompt[len(prompt_prefix):]

                # Create sample for CAPPR
                samples.append(Example(prompt=prompt,
                        completions=ids_in_prompt,
                        prior=prior))
                samples_doc_idx.append(doc_idx)
                samples_retrieved_labels.append(retrieved_labels)
                sample_idx += 1
    
    # Performing CAPPR predictions
    rag_linkage_cappr_predict(col_probs, model_and_tokenizer, samples, samples_doc_idx, samples_retrieved_labels)

    return col_probs

def rag_linkage_construct_prompt(doc_retrievals, document, prompt_template, id_to_label, end_of_seq_str:str):
    # Obtain retrievals grouped by id
    doc_retrievals_dict = {}
    retrieved_ids_counts = {}
    for retrieved in doc_retrievals:
       id = retrieved.metadata["source"]
       content = retrieved.page_content
       doc_retrievals_dict[id] = doc_retrievals_dict.get(id, "") + f"\t{content}\n"
       retrieved_ids_counts[id] = retrieved_ids_counts.get(id, 0) + 1
    
    # List of candidate ids in the prompt
    retrieved_ids = list(doc_retrievals_dict.keys())
    if READER_USE_SHORT_IDS:
        new_doc_retrievals_dict = OrderedDict()
        for idx, text in enumerate(doc_retrievals_dict.values()):
            new_id = READER_SHORT_IDS[idx]
            new_doc_retrievals_dict[new_id] = text
        doc_retrievals_dict = new_doc_retrievals_dict
        ids_in_prompt = list(doc_retrievals_dict.keys())
    else:
        ids_in_prompt = retrieved_ids # Aliasing necessary

        # USE_NO_BK and READER_USE_SHORT_IDS are incompatible
        if USE_NO_BK:
            # Consistently define K candidates            
            n_missing_ids = RETRIEVER_K - len(ids_in_prompt)
            if n_missing_ids > 0:
                # Randomly add missing candidates from the set of possible ids
                ids_in_prompt_set = set(ids_in_prompt)
                all_ids_set = set(id_to_label.keys())
                missing_ids = list(all_ids_set - ids_in_prompt_set)
                new_ids = random.sample(missing_ids, min(n_missing_ids, len(missing_ids)))
                ids_in_prompt += new_ids
                [retrieved_ids_counts.update({id: retrieved_ids_counts.get(id, 0) + 1}) for id in new_ids] # Update counts                
            # Shuffle the list of retrieved ids to avoid order bias
            random.shuffle(ids_in_prompt)

    # Generate the prompt
    if USE_NO_BK: # Prompt without BK
        prompt = prompt_template.format(
            document=document, names=ids_in_prompt
        )
    else: # Prompt with BK (default)
        # Generate background
        background = "".join([f"\nID={id}\n{text}" for id, text in doc_retrievals_dict.items()])
        prompt = prompt_template.format(
            document=document, background=background, names=ids_in_prompt
        )
    
    if READER_REMOVE_ROLE_END_STR:
        prompt = prompt[:-(len(end_of_seq_str)+1)] # Remove the end role/sequence mark

    return prompt, ids_in_prompt, retrieved_ids, retrieved_ids_counts

def rag_linkage_cappr_predict(col_probs, model_and_tokenizer, samples, samples_doc_idx, samples_retrieved_labels):
    pred_probs = predict_proba_examples(samples, model_and_tokenizer=model_and_tokenizer, batch_size=READER_BATCH_SIZE)
    for probs, doc_idx, retrieved_labels in zip(pred_probs, samples_doc_idx, samples_retrieved_labels):
        col_probs[doc_idx][retrieved_labels] = probs

In [17]:
def eval_rag_linkage_df(df:pd.DataFrame, k:int, id_to_label:dict, retrievals:dict, model_and_tokenizer, prompt_template, verbose:bool=True):
    predictions = {}
    top_1_accuracies = {}
    top_k_accuracies = {}
    ids = list(df[ID_KEY])

    for col_name, col_retrievals in retrievals.items():
        if verbose:
            print(f"Evaluation of {col_name} documents")
        
        predictions[col_name] = rag_linkage_col(df[col_name], col_retrievals, model_and_tokenizer, prompt_template, id_to_label)
        top_1_accuracies[col_name], top_k_accuracies[col_name] = eval_rag_linkage_col(ids, predictions[col_name], k, id_to_label)

        if verbose:
            print(f"Top-1 accuracies for {col_name}: {top_1_accuracies[col_name]}")
            print(f"Top-{k} accuracies for {col_name}: {top_k_accuracies[col_name]}")
    
    return top_1_accuracies, top_k_accuracies, predictions   

def eval_rag_linkage_col(ids:List[str], preds:np.ndarray, k:int, id_to_label:dict)->tuple:
  top_1_count = 0
  top_k_count = 0

  for idx, id in enumerate(ids):
    # If there is a prediction
    if preds[idx].sum() != 0:
      label = id_to_label[id]
      sorted_labels = np.argsort(preds[idx])
      top_1_count += 1 if sorted_labels[-1] == label else 0
      top_k_count += 1 if label in sorted_labels[-k:] else 0

  top_1_accuracy = 100 * top_1_count / len(df)
  top_k_accuracy = 100 * top_k_count / len(df)

  return top_1_accuracy, top_k_accuracy

def accuracies_to_csv(accuracy_data: dict, method_name: str, filename: str):
    try:
        # Get the names of the datasets from the dictionary keys.
        # These will serve as the main column headers for the accuracy values.
        dataset_names = list(accuracy_data.keys())

        # Construct the header row for the CSV.
        # The first element is a label for the method name column, followed by dataset names.
        header_row = ['Method'] + dataset_names

        # Construct the data row for the method's accuracies.
        # It starts with the method's name, then appends each accuracy score
        # corresponding to the order of `dataset_names`.
        datetime_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S") 
        data_row = [datetime_str, method_name]
        for dataset in dataset_names:
            # Ensure the accuracy value exists for the dataset
            if dataset in accuracy_data:
                data_row.append(f"{accuracy_data[dataset]:.2f}")
            else:
                # Append an empty string or a placeholder if a dataset is missing
                # (though `dataset_names` is derived from `accuracy_data` keys,
                # this provides robustness if data structures change).
                data_row.append('')

        # Determine if the file exists to decide whether to write headers and append or overwrite.
        file_exists = os.path.exists(filename)

        # Open the CSV file. Use 'w' mode if it's a new file (to write headers),
        # otherwise use 'a' mode (append) if it already exists.
        # `newline=''` is crucial for CSV files to prevent extra blank rows.
        with open(filename, 'a+', newline='', encoding='utf-8') as csvfile:
            # Create a CSV writer object.
            csv_writer = csv.writer(csvfile)

            # Write the header row ONLY if the file did not exist previously.
            if not file_exists:
                csv_writer.writerow(header_row)

            # Write the data row to the CSV
            csv_writer.writerow(data_row)

        print(f"✅ Accuracy data for method '{method_name}' successfully written to '{filename}'.")

    except IOError as e:
        print(f"❌ Error writing to file '{filename}': {e}")
    except Exception as e:
        print(f"❌ An unexpected error occurred: {e}")

## Execution

### Retrievals

In [18]:
id_to_label = OrderedDict([(id, idx) for idx, id in enumerate(df[ID_KEY])])
retrievals = retrieval_df(df, retriever, RETRIEVER_K)

Retrievals for col_name='text':   0%|          | 0/553 [00:00<?, ?it/s]

Retrievals for col_name='St.NER3':   0%|          | 0/553 [00:00<?, ?it/s]

Retrievals for col_name='St.NER4':   0%|          | 0/553 [00:00<?, ?it/s]

Retrievals for col_name='St.NER7':   0%|          | 0/553 [00:00<?, ?it/s]

Retrievals for col_name='spaCy':   0%|          | 0/553 [00:00<?, ?it/s]

...

Retrievals for col_name='k-anonymity_Greedy':   0%|          | 0/553 [00:00<?, ?it/s]

Retrievals for col_name='Manual':   0%|          | 0/553 [00:00<?, ?it/s]

Retrievals for col_name='Student-LLM':   0%|          | 0/553 [00:00<?, ?it/s]

Retrievals for col_name='Sparks-LLM':   0%|          | 0/553 [00:00<?, ?it/s]

Retrievals for col_name='MvM-LLM':   0%|          | 0/553 [00:00<?, ?it/s]

### RAG linkage

In [19]:
print(RAG_LINKAGE_NAME)
rag_top_1_accuracies, rag_top_k_accuracies, rag_predictions = eval_rag_linkage_df(df, RETRIEVER_K, id_to_label, retrievals, model_and_tokenizer, rag_prompt_template)
accuracies_to_csv(rag_top_1_accuracies, RAG_LINKAGE_NAME, RESULTS_FILEPATH)

RAG-Linkage Wiki553_BK=Public k=10_chunk=128_FAISS_NoMasks_thenlper/gte-small unsloth/Qwen3-4B-bnb-4bit_Quant=None_Prompt=5_RemoveEnd=False_CAPPR_Prior=False_Cache=True_ShortIDs=False
Evaluation of text documents


conditional log-probs:   0%|          | 0/463 [00:00<?, ?it/s]

KeyboardInterrupt: 